# Extract oMST REC and LDI scores using subject ID

This notebook scans a JATOS results zip or extracted folder, finds each `data.txt`,
uses the `subject` field inside the TXT as the participant ID,
extracts the final `REC` and `LDI` values from the oMST summary,
and saves a combined CSV.


In [10]:
from pathlib import Path
import zipfile
import re
import pandas as pd

INPUT_PATH = Path('/Users/jianleguo/Documents/GitHub/vmps_mario/data/batch02_new_set_9_participant/oMST/jatos_results_data_20261008035207.zip')
OUTPUT_DIR = Path('/Users/jianleguo/Documents/GitHub/vmps_mario/data/batch02_new_set_9_participant/oMST/data')
OUTPUT_CSV = OUTPUT_DIR / 'combined_oMST_result_by_subject.csv'

SUBJECT_PATTERN = re.compile(r'"subject"\s*:\s*"([^"]+)"')
REC_LDI_PATTERN = re.compile(r'REC,\s*([0-9]*\.?[0-9]+)\s*,\s*LDI,\s*([0-9]*\.?[0-9]+)')


In [11]:
def list_data_files(input_path: Path):
    if input_path.is_file() and input_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(input_path, 'r') as zf:
            return [name for name in zf.namelist() if name.endswith('data.txt')]
    elif input_path.is_dir():
        return sorted(input_path.rglob('data.txt'))
    else:
        raise FileNotFoundError(f'Cannot find input: {input_path}')

def read_text(input_path: Path, file_ref):
    if input_path.is_file() and input_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(input_path, 'r') as zf:
            return zf.read(file_ref).decode('utf-8', errors='replace')
    return Path(file_ref).read_text(encoding='utf-8', errors='replace')

def extract_subject(text: str):
    matches = SUBJECT_PATTERN.findall(text)
    return matches[-1] if matches else None

def extract_rec_ldi(text: str):
    matches = REC_LDI_PATTERN.findall(text)
    if not matches:
        return None
    rec, ldi = matches[-1]
    return float(rec), float(ldi)


In [12]:
rows = []
skipped = []

for file_ref in list_data_files(INPUT_PATH):
    text = read_text(INPUT_PATH, file_ref)
    subject = extract_subject(text)
    score = extract_rec_ldi(text)

    if subject is None or score is None:
        skipped.append(str(file_ref))
        continue

    rec, ldi = score
    rows.append({
        'participant_id': subject,
        'REC': rec,
        'LDI': ldi,
    })

df = pd.DataFrame(rows).drop_duplicates(subset=['participant_id']).sort_values('participant_id').reset_index(drop=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved {len(df)} rows to: {OUTPUT_CSV}')
print(f'Skipped files: {len(skipped)}')
df.head(10)


Saved 9 rows to: /Users/jianleguo/Documents/GitHub/vmps_mario/data/batch02_new_set_9_participant/oMST/data/combined_oMST_result_by_subject.csv
Skipped files: 36


,participant_id,REC,LDI
0,5e6b034bffc1a90c3fe8c234,0.900,0.810
1,5e84bd7c34d5d9072128fac1,0.950,0.712
2,6738bbb49b751f21694bb4af,0.934,0.795
3,673cc0c935b49103f2be1e27,1.000,0.726
4,69602ae6ebc51ed9c9cadbe5,0.800,0.703
5,69fe143fe0fc2630be12c1a4,0.388,0.402
6,6a0a1bf36aaad00ce240dd0f,1.000,0.341
7,6a269ff5749895f01921daf1,0.950,0.514
8,6a983a60a8de63db66503b3f,0.900,0.703


In [13]:
df


,participant_id,REC,LDI
0,5e6b034bffc1a90c3fe8c234,0.900,0.810
1,5e84bd7c34d5d9072128fac1,0.950,0.712
2,6738bbb49b751f21694bb4af,0.934,0.795
3,673cc0c935b49103f2be1e27,1.000,0.726
4,69602ae6ebc51ed9c9cadbe5,0.800,0.703
5,69fe143fe0fc2630be12c1a4,0.388,0.402
6,6a0a1bf36aaad00ce240dd0f,1.000,0.341
7,6a269ff5749895f01921daf1,0.950,0.514
8,6a983a60a8de63db66503b3f,0.900,0.703
